# When does the sigmoid become a step function — and why does it matter?

The steepness of the sigmoid is controlled by θ₁:

$$P(y=1 | x) = \frac{1}{1 + e^{-(\theta_0 + \theta_1 x)}}$$

As |θ₁| → ∞, the sigmoid approaches a **step function** — the model becomes completely certain on both sides of the decision boundary.

There are **two distinct routes** to a step-like sigmoid, and they have very different implications:

1. **True separation** — the classes genuinely do not overlap in the population
2. **Accidental separation** — the classes overlap in the population, but a small sample happened to come out separable by chance

We will show both cases and explain why the step-like result is problematic in each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.linear_model import LogisticRegression
from scipy.special import erfc
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

def generate_data(n_per_class=10, class_sep=1.0, x_spread=1.0, seed=0):
    rng = np.random.RandomState(seed)
    X0 = rng.normal(loc=-class_sep/2, scale=x_spread, size=n_per_class)
    X1 = rng.normal(loc=+class_sep/2, scale=x_spread, size=n_per_class)
    X  = np.concatenate([X0, X1]).reshape(-1, 1)
    y  = np.array([0]*n_per_class + [1]*n_per_class)
    return X, y

def fit_logistic(X, y, C=1e4):
    clf = LogisticRegression(C=C, solver='lbfgs', max_iter=10000)
    clf.fit(X, y)
    return clf.intercept_[0], clf.coef_[0, 0]

def sigmoid(x, t0, t1):
    return 1 / (1 + np.exp(-np.clip(t0 + t1*x, -100, 100)))

def overlap_pct(sep, spread):
    return 100 * erfc(sep / (2 * spread * np.sqrt(2)))

x_plot = np.linspace(-12, 12, 500)
SEED   = 7
print("Ready.")

## Case 1 — True separation, varying spread

Here the classes genuinely do not overlap: `sep = 10 × σ` in every panel, giving SNR = 10 and ~0% population overlap.

As σ increases (wider spread), θ₁ decreases — even though the classes are equally well separated in SNR terms.
The sigmoid remains step-like in all panels, but the **transition width** in x-space grows with σ,
because the step must span the larger spread of the data.

In [ ]:
spreads = [0.3, 0.7, 1.2, 2.0, 3.5]

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
fig.suptitle('Case 1: True separation  (sep = 10×σ → SNR = 10 → overlap ≈ 0% in all panels)\n'
             'Wider spread → smaller θ₁, but sigmoid stays step-like',
             fontsize=12, fontweight='bold')

for ax, spread in zip(axes, spreads):
    sep = 10 * spread
    X, y = generate_data(10, sep, spread, seed=SEED)
    t0, t1 = fit_logistic(X, y)

    ax.plot(x_plot, sigmoid(x_plot, t0, t1), color='darkred', linewidth=2.2, zorder=2)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)

    rng = np.random.RandomState(SEED + int(spread*10))
    j0 = rng.uniform(-0.04, 0.04, (y==0).sum())
    j1 = rng.uniform(-0.04, 0.04, (y==1).sum())
    ax.scatter(X[y==0], j0,     color='royalblue', s=50, alpha=0.9, zorder=3, label='class 0')
    ax.scatter(X[y==1], 1 + j1, color='tomato',    s=50, alpha=0.9, zorder=3, label='class 1')

    ov = overlap_pct(sep, spread)
    ax.set_title(f'σ={spread}, sep={sep:.1f}\noverlap={ov:.2f}%', fontsize=9)
    ax.annotate(f'θ₁={t1:.1f}', xy=(0.97, 0.52), xycoords='axes fraction',
                ha='right', fontsize=10, color='darkred', fontweight='bold')
    ax.set_xlim(-12, 12)
    ax.set_ylim(-0.15, 1.15)
    ax.set_yticks([0, 0.5, 1])
    ax.set_xlabel('x', fontsize=9)
    ax.grid(True, axis='x', alpha=0.25)

axes[0].set_ylabel('P(y=1|x)', fontsize=9)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig('case1_true_separation.png', dpi=130, bbox_inches='tight')
plt.show()

## Case 2 — Accidental separation from a small sample

Here the **population** has large overlap (`sep=1.0, σ=1.0 → ~32%`), but with only 10 points per class,
some random draws happen to produce a perfectly separable sample.

The model sees no overlap, pushes θ₁ to a large value, and produces a step-like sigmoid —
even though the true underlying distributions heavily overlap.

In [ ]:
SEP_POP    = 1.0
SPREAD_POP = 1.0

# find seeds where the sample is accidentally separable
lucky_seeds = []
for seed in range(1000):
    X, y = generate_data(10, SEP_POP, SPREAD_POP, seed=seed)
    if X[y==0].max() < X[y==1].min() or X[y==1].max() < X[y==0].min():
        lucky_seeds.append(seed)
    if len(lucky_seeds) == 3:
        break

print(f"Found {len(lucky_seeds)} accidentally separable samples in first 1000 draws")
print(f"Seeds: {lucky_seeds}")
print(f"Population overlap: {overlap_pct(SEP_POP, SPREAD_POP):.1f}%")

In [ ]:
seeds_to_plot = [0] + lucky_seeds   # seed 0 = typical overlapping sample
titles = ['Typical sample\n(overlapping, as expected)'] + \
         [f'Lucky draw (seed={s})\naccidentally separable' for s in lucky_seeds]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=True)
fig.suptitle(f'Case 2: Accidental separation  (population overlap = {overlap_pct(SEP_POP, SPREAD_POP):.0f}%,  n = 10 per class)\n'
              'Small samples can be accidentally separable → step-like sigmoid despite high population overlap',
             fontsize=12, fontweight='bold')

for ax, seed, title in zip(axes, seeds_to_plot, titles):
    X, y = generate_data(10, SEP_POP, SPREAD_POP, seed=seed)
    t0, t1 = fit_logistic(X, y)

    ax.plot(x_plot, sigmoid(x_plot, t0, t1), color='darkred', linewidth=2.2, zorder=2)
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.6)

    rng = np.random.RandomState(seed + 999)
    j0 = rng.uniform(-0.04, 0.04, (y==0).sum())
    j1 = rng.uniform(-0.04, 0.04, (y==1).sum())
    ax.scatter(X[y==0], j0,     color='royalblue', s=55, alpha=0.9, zorder=3, label='class 0')
    ax.scatter(X[y==1], 1 + j1, color='tomato',    s=55, alpha=0.9, zorder=3, label='class 1')

    ax.set_title(title, fontsize=9)
    ax.annotate(f'θ₁={t1:.1f}', xy=(0.97, 0.52), xycoords='axes fraction',
                ha='right', fontsize=10, color='darkred', fontweight='bold')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-0.15, 1.15)
    ax.set_yticks([0, 0.5, 1])
    ax.set_xlabel('x', fontsize=9)
    ax.grid(True, axis='x', alpha=0.25)

axes[0].set_ylabel('P(y=1|x)', fontsize=9)
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.06))
plt.tight_layout()
plt.savefig('case2_accidental_separation.png', dpi=130, bbox_inches='tight')
plt.show()

## Why is the step-like sigmoid problematic?

The two cases above require different answers.

---

### Case 1 (true separation): a *training* problem, not a modeling problem

The step function is actually the **correct answer** — the model is right to be certain. The problem is that the MLE has no finite solution.
The cross-entropy loss keeps decreasing as |θ₁| → ∞ but **never reaches a minimum**:

$$\mathcal{L}(\theta) = -\sum_{i} \left[ y_i \log \sigma(\theta^\top x_i) + (1-y_i) \log(1 - \sigma(\theta^\top x_i)) \right]$$

The optimizer never converges. Weights grow without bound. The solution is **L2 regularization**, which adds a penalty term that creates a finite optimum:

$$\mathcal{L}_{\text{reg}}(\theta) = \mathcal{L}(\theta) + \frac{\lambda}{2} \|\theta\|^2$$

This pulls θ₁ back from infinity and gives a well-defined, steep (but finite) sigmoid. The step-like behavior is preserved; the training problem is solved.

---

### Case 2 (accidental separation): a *generalization* problem

Here the step-like sigmoid is **wrong**. The model is confidently predicting near-0 or near-1 everywhere, but the true population has 32% overlap — meaning a new test point in the overlap region genuinely could belong to either class, and the model should reflect that uncertainty with a probability close to 0.5, not 0 or 1.

The consequences:
- **Overconfident predictions**: the model assigns near-zero probability to outcomes that are actually quite plausible
- **Poor calibration**: the predicted probabilities do not reflect actual frequencies — a prediction of 0.99 does not mean the event happens 99% of the time
- **Bad generalization**: the model memorized the accident of this particular small sample, not the underlying pattern

Again, **L2 regularization** is the fix — but for a different reason. Here it is not just a computational necessity but a genuine prior that prevents the model from overreacting to small samples.

---

### Summary

| | Cause | Step sigmoid is... | Problem | Fix |
|---|---|---|---|---|
| Case 1 | True separation | Correct | MLE has no finite solution | L2 regularization (computational) |
| Case 2 | Accidental separation (small n) | Wrong | Overconfident, poor calibration | L2 regularization (modeling) |

## Demonstration: regularization rescues both cases

In [ ]:
C_values  = [1e4, 1.0, 0.1]
C_labels  = ['C=10⁴  (no reg)', 'C=1  (moderate)', 'C=0.1  (strong)']
colors_C  = ['#d62728', '#e07b39', '#1f77b4']

# Case 1: true separation
X1, y1 = generate_data(10, sep=3.0, x_spread=0.3, seed=SEED)
# Case 2: accidental separation
X2, y2 = generate_data(10, sep=1.0, x_spread=1.0, seed=lucky_seeds[0])

fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
fig.suptitle('Effect of L2 regularization on step-like sigmoids\n'
             'Top: true separation  |  Bottom: accidental separation',
             fontsize=12, fontweight='bold')

row_data  = [(X1, y1, 'Case 1: true separation\n(σ=0.3, sep=3.0, overlap≈0%)'),
             (X2, y2, f'Case 2: accidental separation\n(σ=1.0, sep=1.0, population overlap=32%)')]

for r, (X, y, row_label) in enumerate(row_data):
    for c, (C, cl, col) in enumerate(zip(C_values, C_labels, colors_C)):
        ax = axes[r, c]
        t0, t1 = fit_logistic(X, y, C=C)

        ax.plot(x_plot, sigmoid(x_plot, t0, t1), color=col, linewidth=2.2, zorder=2)
        ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.7, alpha=0.5)

        rng = np.random.RandomState(42)
        j0 = rng.uniform(-0.04, 0.04, (y==0).sum())
        j1 = rng.uniform(-0.04, 0.04, (y==1).sum())
        ax.scatter(X[y==0], j0,     color='royalblue', s=50, alpha=0.9, zorder=3)
        ax.scatter(X[y==1], 1 + j1, color='tomato',    s=50, alpha=0.9, zorder=3)

        ax.annotate(f'θ₁={t1:.1f}', xy=(0.97, 0.52), xycoords='axes fraction',
                    ha='right', fontsize=10, color=col, fontweight='bold')
        ax.set_xlim(-8, 8)
        ax.set_ylim(-0.15, 1.15)
        ax.set_yticks([0, 0.5, 1])
        ax.grid(True, axis='x', alpha=0.25)
        ax.set_xlabel('x', fontsize=9)

        if r == 0:
            ax.set_title(cl, fontsize=10, color=col, fontweight='bold')
        if c == 0:
            ax.set_ylabel(row_label + '\nP(y=1|x)', fontsize=8)

plt.tight_layout()
plt.savefig('regularization_effect.png', dpi=130, bbox_inches='tight')
plt.show()